# Task 5 — Step 3: Out-of-Domain SVAMP Transfer & Feedback Synthesis

### Objectives
1. **Out-of-Domain Transfer**: Evaluate SFT, RLVR, and RLAIF policies on the fixed 100-example SVAMP transfer set (`data/math_transfer_eval.jsonl`) without retraining.
2. **Transfer Retention**: Measure performance drop $\Delta_{\text{transfer}} = \text{Acc}_{\text{GSM}} - \text{Acc}_{\text{SVAMP}}$ and retention ratio.
3. **Feedback Synthesis**: Compare RLVR and RLAIF across coverage, noise, exploitability, and compute cost.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Transfer Evaluation & Synthesis


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer
    !python -m task5_feedback.compare_feedback --config configs/feedback.yaml
else:
    print("[OK] Loading existing transfer and feedback synthesis results...")


## 2. In-Domain vs. Out-of-Domain Transfer Table


In [ ]:
syn_file = "results/task5_feedback/feedback_synthesis.json"
if os.path.exists(syn_file):
    with open(syn_file, encoding='utf-8') as f:
        syn_data = json.load(f)
        
    gsm = syn_data.get("in_domain_gsm", {})
    svamp = syn_data.get("transfer_svamp", {})
    ret = syn_data.get("retention_ratios", {})
    
    rows = []
    for p in ["sft", "rlvr", "rlaif"]:
        g_acc = gsm.get(p, {}).get("exact_accuracy", 0)*100
        t_acc = svamp.get(p, {}).get("exact_accuracy", 0)*100
        drop = g_acc - t_acc
        rows.append({
            "Policy": p.upper(),
            "GSM8K In-Domain (%)": f"{g_acc:.1f}%",
            "SVAMP Out-of-Domain (%)": f"{t_acc:.1f}%",
            "Transfer Drop (Δ)": f"{drop:+.1f}%",
            "Retention Ratio": f"{ret.get(p, 0)*100:.1f}%",
            "Transfer Tokens": f"{svamp.get(p, {}).get('mean_tokens', 0):.1f}"
        })
    df_trans = pd.DataFrame(rows)
    display(df_trans)


## 3. Visualizing In-Domain vs. Transfer Retention


In [ ]:
if os.path.exists(syn_file):
    pols = [r["Policy"] for r in rows]
    g_vals = [float(r["GSM8K In-Domain (%)"].replace('%','')) for r in rows]
    t_vals = [float(r["SVAMP Out-of-Domain (%)"].replace('%','')) for r in rows]
    
    x = np.arange(len(pols))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - width/2, g_vals, width, label='GSM8K (In-Domain)', color='#3B82F6', alpha=0.85)
    ax.bar(x + width/2, t_vals, width, label='SVAMP (Transfer)', color='#10B981', alpha=0.85)
    ax.set_ylabel('Exact Answer Accuracy (%)')
    ax.set_title('Generalization and Transfer Degradation (GSM8K -> SVAMP)')
    ax.set_xticks(x)
    ax.set_xticklabels(pols)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## 4. Publication Dashboard Inspection


In [ ]:
p = "report/figures/task5_feedback_synthesis_dashboard.png"
if os.path.exists(p):
    print(f"Displaying publication figure: {p}")
    display(Image(filename=p, width=700))


## 5. Comprehensive Feedback Source Trade-Off Matrix

| Dimension | Exact Verifiable Reward (RLVR) | AI Feedback Preferences (RLAIF) |
|---|---|---|
| **Coverage** | Narrow: requires checkable ground-truth (e.g. math/code) | Broad: applicable to open-ended dialogue, essays, safety |
| **Noise & Variance** | Zero label noise (deterministic program) | Inherits judge model biases, style preferences, ties |
| **Exploitability** | Corrupted intermediate reasoning with lucky final answer | Susceptible to verbose fluff, persuasive tone, length hacking |
| **Compute Cost** | Extremely lightweight (pure CPU string parser) | Heavy: requires full LLM forward passes for pairwise judging |
